# VMMC Windowing — Free Energy Profile of an 8-nt DNA Duplex

**Virtual-Move Monte Carlo (VMMC)** is a cluster-move algorithm that is highly efficient for simulating nucleic acid folding and unfolding.  Unlike molecular dynamics, VMMC proposes coordinated rigid-body moves of nucleotide clusters, dramatically reducing the time needed to cross free-energy barriers.

**Windowing** divides the reaction-coordinate state space into overlapping regions and runs an independent VMMC simulation inside each one.  Every region of the free-energy landscape — including rarely-visited transition states — is sampled with equal statistics.  The per-window histograms are then combined using WHAM to reconstruct the global free-energy profile.

---

### System: 8-nucleotide DNA duplex

```
5' — 0  1  2  3  4  5  6  7 — 3'   strand 1
     |  |  |  |  |  |  |  |
3' — 15 14 13 12 11 10  9  8 — 5'  strand 2
```

**Reaction coordinate:** number of native Watson–Crick hydrogen bonds (0 = fully melted, 8 = fully formed).

| Window | States (bonds) | Starting conf |
|--------|----------------|---------------|
| 0 | 5 – 8 (folded end) | `duplex_box_30.dat` (8 bonds) |
| 1 | 0 – 5 (melted end) | `melted_start.dat` (≤7 bonds) |

The windows overlap at state 5, which is required for WHAM stitching.

> **Note:** Production runs need at least 1 × 10⁷ – 1 × 10⁸ steps per window for converged free-energy estimates.  The `steps` value below is set to 1 × 10⁴ so the notebook runs quickly for demonstration.

In [ ]:
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from ipy_oxdna.utils.order_parameter import OrderParameter
from ipy_oxdna.vmmc_umbrella.windowing import VmmcWindowing

## 1. Paths and file setup

In [ ]:
HERE = Path(".").resolve()
OXDNA_FILES = HERE / "oxdna_files"
OUTPUT_DIR  = HERE / "vmmc_windowing_output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Window 1 needs a starting conf that is already in the melted region.
# The default duplex_box_30.dat starts at 8 bonds (outside window 1's state space).
MELTED_START_DIR = OXDNA_FILES / "melted_start_dir"
MELTED_START_DIR.mkdir(exist_ok=True)
shutil.copy(OXDNA_FILES / "duplex_box_30.top", MELTED_START_DIR)
shutil.copy(OXDNA_FILES / "melted_start.dat", MELTED_START_DIR / "duplex_box_30.dat")

print("oxDNA files:", list(OXDNA_FILES.glob("duplex_box_30.*")))
print("Melted start dir:", list(MELTED_START_DIR.iterdir()))

## 2. Define the order parameter

An `OrderParameter` of type `"bond"` counts Watson–Crick hydrogen bonds between specified nucleotide pairs.  Strand 1 (5'→3') is nucleotides 0–7; strand 2 (3'→5') is nucleotides 8–15, so the Watson–Crick pairs are (0, 15), (1, 14), … (7, 8).

In [ ]:
BONDS = list(zip(
    [0, 1, 2, 3, 4, 5, 6, 7],
    reversed([8, 9, 10, 11, 12, 13, 14, 15]),
))
NATIVE_OP = OrderParameter("native", "bond", BONDS)

print(f"Bond pairs:      {NATIVE_OP.pairs}")
print(f"len(NATIVE_OP):  {len(NATIVE_OP)}   (= n_pairs + 1 = number of possible bond counts)")
print(f"Valid states:    0 … {len(NATIVE_OP) - 1} bonds")

## 3. Define the window state spaces

Each state is a 1-tuple `(n_bonds,)` because we have a single order parameter.  Windows **must overlap** — WHAM requires at least one state shared between adjacent windows.

In [ ]:
WINDOW_0_STATES = {(s,) for s in range(5, 9)}   # 5, 6, 7, 8 bonds  (folded end)
WINDOW_1_STATES = {(s,) for s in range(0, 6)}   # 0, 1, 2, 3, 4, 5 bonds  (melted end)

overlap = WINDOW_0_STATES & WINDOW_1_STATES
print(f"Window 0 states: {sorted(WINDOW_0_STATES)}")
print(f"Window 1 states: {sorted(WINDOW_1_STATES)}")
print(f"Overlap:         {sorted(overlap)}  ← WHAM requires this to be non-empty")

## 4. Build the VmmcWindowing object

Two known quirks to work around:

1. **`filter_legal_states`** must return a `list`, not a `set`.  Override the default with `sorted()`.
2. **`build_start_weights`** is called as `(sim, window_idx)` but the default helper takes only one argument.  Provide a two-argument replacement.

In [ ]:
w = VmmcWindowing(OUTPUT_DIR)
w.add_order_parameter(NATIVE_OP)
w.n_reps = 2   # replicas per window — increase for production

# Temperatures at which the reweighted (unbiased) histograms are computed
w.extrapolate_hist_Ts = ["30C", "37C", "40C", "46C", "55C"]

# Quirk 1
w.filter_legal_states = lambda states: sorted(states)

# Simulation setup — called once per replica before the run
def build_replica(windowing, sim):
    sim.build(clean_build="force")
    sim.input.swap_default_input("vmmc")
    sim.input["steps"]               = int(1e4)   # ← increase for production (1e7–1e8)
    sim.input["T"]                    = "40C"
    sim.input["salt_concentration"]   = 1.0        # 1 M NaCl
    sim.input["interaction_type"]     = "DNA2"
    sim.input["print_energy_every"]   = int(1e3)
    sim.input["print_conf_interval"]  = int(1e3)

w.build_replica = build_replica

# Quirk 2
def build_start_weights(sim, window_idx):
    # generate_weights(T_scale) sets initial weights so the simulation samples
    # all states within the window with roughly equal probability.
    sim.weights[...] = sim.generate_weights(7.0)

w.build_start_weights = build_start_weights

# Register the windows (order matters: window 0 first)
w.add_window(WINDOW_0_STATES, OXDNA_FILES)    # folded end
w.add_window(WINDOW_1_STATES, MELTED_START_DIR)  # melted end

print(f"VmmcWindowing configured: {len(w)} windows, {w.n_reps} replica(s) each")

## 5. Setup

`setup()` builds the simulation directories, writes the oxDNA input files, order-parameter files, and initial weight files for every replica in every window.

In [ ]:
w.setup()

# Inspect what was created
for i, window in enumerate(w):
    sim = window[0]
    print(f"Window {i} replica 0 sim_dir: {sim.sim_dir}")
    print(f"  op_file:      {sim.sim_dir / sim.input['op_file']}")
    print(f"  weights_file: {sim.sim_dir / sim.input['weights_file']}")

## 6. Run

`run(join=True)` blocks until all simulations complete.  Each replica writes a `last_hist_file` containing the visited-state histogram that will be used in the WHAM analysis.

In [ ]:
w.run(join=True)
print("All simulations complete.")

## 7. Read VMMC data and compute statistics

In [ ]:
for window in w:
    for sim in window:
        sim.analysis.read_vmmc_op_data()
        sim.analysis.calculate_sampling_and_probabilities()

# Print per-window sampling statistics
for i, window in enumerate(w):
    print(f"\n=== Window {i} ===")
    for j, sim in enumerate(window):
        print(f"  Replica {j}:")
        print(sim.analysis.statistics[["sampling_percent", "wt_prob"]].to_string())

## 8. WHAM — global free-energy profile

`wham(op=0)` runs the Weighted Histogram Analysis Method across all windows to compute the unbiased probability distribution over the reaction coordinate.

In [ ]:
rho = w.wham(op=0)

states = sorted(rho.keys())
probs  = [rho[s] for s in states]

# Convert probability to free energy: G = -kT ln(p), shifted so min G = 0
G = -np.log(np.array(probs))
G -= G.min()

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(states, G, "o-", lw=2)
ax.set_xlabel("Number of native bonds")
ax.set_ylabel(r"$\Delta G / k_\mathrm{B}T$")
ax.set_title("Free-energy profile — 8-nt duplex at 40 °C")
ax.set_xticks(states)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Visualisation helpers

`VmmcWindowing` provides several built-in plots.

In [ ]:
# Per-window weight profiles
fig, axes = w.plot_window_weights()
fig.suptitle("Per-window VMMC weights")
fig.tight_layout()
plt.show()

In [ ]:
# Per-window sampling histograms
fig, _ = w.plot_window_data(NATIVE_OP)
fig.suptitle("Per-window sampling histograms")
fig.tight_layout()
plt.show()

In [ ]:
# Built-in free-energy profile (calls wham internally)
fig, ax = w.plot_free_energy_profile()
ax.set_title("Built-in free-energy profile")
plt.tight_layout()
plt.show()

In [ ]:
# State histograms for each window
for i, window in enumerate(w):
    fig = window.create_state_histograms(NATIVE_OP)
    if fig is not None:
        fig.suptitle(f"Window {i} — state histograms")
        fig.tight_layout()
        plt.show()

## 10. Resuming a completed run

`setup()` writes a `setup.json` cache file to the output directory.  In a later session you can reconstruct the full object without re-running simulations:

```python
w = VmmcWindowing(OUTPUT_DIR)
w.load()
for window in w:
    for sim in window:
        sim.analysis.read_vmmc_op_data()
        sim.analysis.calculate_sampling_and_probabilities()
rho = w.wham(op=0)
```

## 11. Saving merged weights for the next iteration

If you plan to run multiple reweighting iterations, save the merged weight profile so it can be used as the starting point for the next round:

```python
w.save_merged_weights(fname="merged_weights.txt")
# load in the next run with:
# sim.input['weights_file'] = 'merged_weights.txt'
```